# Step 06: RF-DETR Multi-Class Prediction & Sample Visual Inspector
### High-Throughput Test Inference & Recalibrated Sample Prediction Showcase
Directly extracted and extended from `multi_class_train_rfdetr.ipynb` (Cells 1, 2, 14, 21, 22).

### Workflow:
1. Auto-discovers fine-tuned model weights from `model_registry.json` or `model/`.
2. Runs batch test inference with **Pure PIL & IPython visualization** (zero matplotlib disk leaks).
3. Applies **Boundary-Box Shrinkage** (`shrink_factor=0.96`, 4% margin contraction) to tightly frame labels.
4. Uses **Sample Visual Badges** (`use_textline_orientation=True`) to read location tag shelf labels.
5. Displays **sample visual predictions** with color-coded bounding boxes and confidence badges.
6. Saves predictions JSON to `predictions/`.


In [ ]:
# STEP 0 - Environment Dependencies
%pip install -q pycocotools Pillow opencv-python-headless==4.9.0.80 tabulate


In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
import urllib.parse
from urllib.parse import urlparse, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# CELL 2 - Model Weights Auto-Discovery & Path Resolution
REPO_ROOT = next((p.resolve() for p in [Path("."), Path(".."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path(".").resolve())
PIPELINE_NAME = "multi_class_rfdetr"
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME
INPUT_JSON_DIR = Path("coco_files") if Path("coco_files").exists() else (REPO_ROOT / "coco_files")
MODEL_VERSION = os.getenv("MODEL_VERSION", "v2")
VERSION_TAG = MODEL_VERSION if MODEL_VERSION else "v1"
VERSION_DIR = PIPELINE_DIR / VERSION_TAG
candidate_model_dirs = [
    VERSION_DIR / "model",
    VERSION_DIR / "checkpoints",
    REPO_ROOT / "multi_class_train_rfdetr" / "model",
    Path("model")
]
FINAL_MODEL_DIR = next((p for p in candidate_model_dirs if p.exists() and any(p.glob("*.pth"))), candidate_model_dirs[0])

candidates = list(FINAL_MODEL_DIR.glob("*.pth"))
BEST_MODEL_PATH = None
for c in candidates:
    if "precision" in c.name:
        BEST_MODEL_PATH = c
        break
if not BEST_MODEL_PATH and candidates:
    BEST_MODEL_PATH = candidates[0]

candidate_dataset_dirs = [
    PIPELINE_DIR / "dataset",
    PIPELINE_DIR / "dataset_full_data",
    VERSION_DIR / "dataset",
    REPO_ROOT / "dataset",
    REPO_ROOT / "multi_class_rfdetr" / "dataset",
    Path("/home/jupyter/multi_class_rfdetr/dataset"),
    REPO_ROOT / "multi_class_train_rfdetr" / "dataset_full_data",
    Path("dataset_stratified")
]
DATASET_DIR = next((p for p in candidate_dataset_dirs if (p / "test" / "_annotations.coco.json").exists()), candidate_dataset_dirs[0])

TEST_ANN = str(DATASET_DIR / "test" / "_annotations.coco.json")
IMAGES_DIR = REPO_ROOT / "coco_files" / "images"
if not IMAGES_DIR.exists():
    IMAGES_DIR = DATASET_DIR / "images"

INFERENCE_OUTPUT_DIR = Path("predictions")
INFERENCE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIDENCE = 0.50
NMS_THRESHOLD = 0.50
RESOLUTION = 560
SHRINK_FACTOR = 0.96

logger.info(f"Model Path:         {BEST_MODEL_PATH}")
logger.info(f"Test Annotation:    {TEST_ANN}")


In [ ]:
# CELL 14 - Safe Pretrained Weights Loading & Detection Head Patch
from rfdetr.models.lwdetr import LWDETR

# 1. Patch LWDETR.load_state_dict using torch.nn.Module.load_state_dict directly (prevents RecursionError)
def _safe_lwdetr_load_state_dict(self, state_dict, strict=True):
    """Filters out classification heads (class_embed, enc_out_class_embed) from checkpoint when num_classes differs."""
    model_state = self.state_dict()
    filtered_state_dict = {}
    mismatched = []
    
    for k, v in state_dict.items():
        clean_k = k
        if clean_k not in model_state and clean_k.startswith("model."):
            clean_k = clean_k[6:]
        if clean_k not in model_state and clean_k.startswith("module."):
            clean_k = clean_k[7:]
            
        if clean_k in model_state:
            if model_state[clean_k].shape == v.shape:
                filtered_state_dict[clean_k] = v
            else:
                mismatched.append((clean_k, tuple(v.shape), tuple(model_state[clean_k].shape)))
        elif k in model_state:
            if model_state[k].shape == v.shape:
                filtered_state_dict[k] = v
            else:
                mismatched.append((k, tuple(v.shape), tuple(model_state[k].shape)))

    if mismatched:
        logger.info(f"Notice: Filtered {len(mismatched)} classification head layers with class-count mismatch from COCO weights:")
        for name, ckpt_shape, model_shape in mismatched[:4]:
            logger.info(f"   - {name}: checkpoint {ckpt_shape} -> target model {model_shape}")
        if len(mismatched) > 4:
            logger.info(f"   - ... and {len(mismatched) - 4} more classification head tensors.")
        logger.info("Pretrained backbone, transformer, and regression heads loaded successfully.")
    
    # Call base PyTorch nn.Module.load_state_dict directly to avoid any recursion on re-runs
    return torch.nn.Module.load_state_dict(self, filtered_state_dict, strict=False)

LWDETR.load_state_dict = _safe_lwdetr_load_state_dict
logger.info("LWDETR.load_state_dict patched for safe custom num_classes loading.")

# 2. Patch reinitialize_detection_head to handle both class_embed and transformer.enc_out_class_embed
def _fixed_reinitialize(self, num_classes):
    lwdetr = self.model  # the actual LWDETR nn.Module
    device = next(lwdetr.parameters()).device
    
    # Reinitialize class_embed
    if hasattr(lwdetr, "class_embed"):
        in_feat = lwdetr.class_embed.in_features
        lwdetr.class_embed = nn.Linear(in_feat, num_classes).to(device)
        nn.init.normal_(lwdetr.class_embed.weight, std=0.01)
        nn.init.zeros_(lwdetr.class_embed.bias)
        
    # Reinitialize transformer.enc_out_class_embed
    if hasattr(lwdetr, "transformer") and hasattr(lwdetr.transformer, "enc_out_class_embed"):
        enc_heads = lwdetr.transformer.enc_out_class_embed
        if isinstance(enc_heads, nn.ModuleList):
            for i in range(len(enc_heads)):
                in_feat = enc_heads[i].in_features
                enc_heads[i] = nn.Linear(in_feat, num_classes).to(device)
                nn.init.normal_(enc_heads[i].weight, std=0.01)
                nn.init.zeros_(enc_heads[i].bias)
        elif isinstance(enc_heads, nn.Linear):
            in_feat = enc_heads.in_features
            lwdetr.transformer.enc_out_class_embed = nn.Linear(in_feat, num_classes).to(device)
            nn.init.normal_(lwdetr.transformer.enc_out_class_embed.weight, std=0.01)
            nn.init.zeros_(lwdetr.transformer.enc_out_class_embed.bias)
            
    logger.info(f"Detection heads (class_embed + enc_out_class_embed) initialized for num_classes={num_classes}")

rfdetr_main.Model.reinitialize_detection_head = _fixed_reinitialize
logger.info("reinitialize_detection_head patched.")

# 3. Model Instantiation
model_cls_map = {
    "nano": RFDETRNano,
    "small": RFDETRSmall,
    "medium": RFDETRMedium,
    "base": RFDETRBase,
    "large": RFDETRLarge
}
ModelClass = model_cls_map.get(MODEL_SIZE.lower(), RFDETRBase)
model = ModelClass(
    num_classes=NUM_CLASSES,
    pretrain_weights=PRETRAINED_WEIGHTS,
    resolution=RESOLUTION
)
logger.info(f"RF-DETR-{MODEL_SIZE.capitalize()} instantiated with resolution={RESOLUTION}, num_classes={NUM_CLASSES}")

# Explicitly ensure detection heads match NUM_CLASSES
if hasattr(model, "model") and hasattr(model.model, "reinitialize_detection_head"):
    model.model.reinitialize_detection_head(NUM_CLASSES)

# Build ordered list of class names from COCO categories
class_names = [
    train_info["categories"][cid]
    for cid in sorted(train_info["categories"].keys())
]
logger.info(f"class_names: {class_names}")

In [ ]:
# CELL 21 - Test Set Inference & Visual Predictions (Pure PIL & IPython - Zero Matplotlib Dependency)
from PIL import Image
from IPython.display import Image as IPImage, display

logger.info("=" * 70)
logger.info(f"[CELL 21] Running inference on test split (Confidence={CONFIDENCE}, NMS={NMS_THRESHOLD})...")
logger.info("=" * 70)

import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass
try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

test_ds = COCODetectionDataset(str(IMAGES_DIR), TEST_ANN, RESOLUTION)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

test_map_results = compute_map(lwdetr, test_loader, device, img_size=RESOLUTION, threshold=CONFIDENCE_THRESHOLD)
logger.info("Test Set mAP Results:")
for k, v in test_map_results.items():
    if isinstance(v, torch.Tensor):
        if v.numel() == 1:
            logger.info(f"   - {k:20s}: {v.item():.4f}")
        else:
            logger.info(f"   - {k:20s}: {[round(x, 4) for x in v.tolist()]}")

# Visual sample test predictions using supervision + PIL save/display
color_palette = sv.ColorPalette.from_hex([
    "#e6194B", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990"
])
box_annotator = sv.BoxAnnotator(color=color_palette, thickness=2)
label_annotator = sv.LabelAnnotator(color=color_palette, text_scale=0.5, text_thickness=1)

os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)
lwdetr.eval()
preview_count = 0

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Inference batches", unit="batch", leave=False):
        if preview_count >= NUM_INFERENCE_PREVIEWS:
            break
        images = images.to(device)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            outputs = lwdetr(images)
        pred_logits = outputs["pred_logits"][0]
        pred_boxes = outputs["pred_boxes"][0]

        scores, class_ids = pred_logits.sigmoid().max(-1)
        keep = scores > CONFIDENCE
        if keep.sum() == 0:
            continue

        boxes_xyxy = (box_cxcywh_to_xyxy(pred_boxes[keep]) * RESOLUTION).cpu().numpy()
        scores_np = scores[keep].cpu().numpy()
        class_ids_np = class_ids[keep].cpu().numpy()

        img_np = (images[0].permute(1, 2, 0).cpu().numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)

        detections = sv.Detections(xyxy=boxes_xyxy, confidence=scores_np, class_id=class_ids_np)
        labels = [f"{class_names[cid] if cid < len(class_names) else str(cid)} {c:.2f}" for cid, c in zip(class_ids_np, scores_np)]
        annotated = box_annotator.annotate(scene=img_np.copy(), detections=detections)
        annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

        # Save annotated image directly and display via IPython (no Matplotlib)
        out_img_path = os.path.join(INFERENCE_OUTPUT_DIR, f"preview_detection_{preview_count+1}.png")
        Image.fromarray(annotated).save(out_img_path)
        logger.info(f"Preview #{preview_count+1}: {len(boxes_xyxy)} objects detected -> Saved to {out_img_path}")
        display(IPImage(filename=out_img_path))
        preview_count += 1

logger.info(f"Successfully generated and previewed {preview_count} test detection images.")

In [ ]:
# CELL 6 - Sample Predictions Visual Gallery with Multi-Class Detection Badges
logger.info("Rendering Gallery of Sample Model Predictions:")
for pred in predictions_list[:5]:
    img_p = IMAGES_DIR / pred["image_name"]
    if not img_p.exists(): continue
    pil_im = Image.open(img_p).convert("RGB")
    draw = ImageDraw.Draw(pil_im)
    
    for box in pred.get("boxes", []):
        x1, y1, x2, y2 = box["box_2d"]
        cat_name = box["category_name"]
        score = box["score"]
        color = "#00E676" if cat_name == "location_tag" else ("#2979FF" if "aisle" in cat_name.lower() else "#00B0FF")
        
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        badge_text = f"{cat_name} ({score:.2f})"
        draw.rectangle([x1, max(0, y1-20), x1 + len(badge_text)*8, y1], fill=color)
        draw.text((x1+4, max(2, y1-16)), badge_text, fill="#000000")
        
    display(pil_im.resize((pil_im.width // 2, pil_im.height // 2)))


In [ ]:
# CELL 22 - Save Test Predictions JSON
logger.info("=" * 70)
logger.info("[CELL 22] Saving test predictions...")
logger.info("=" * 70)

pred_out_file = os.path.join(INFERENCE_OUTPUT_DIR, f"test_predictions_{RUN_ID}.json")
lwdetr.eval()
all_test_predictions = {}

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Inference batches", unit="batch", leave=False):
        img_id = targets[0]["image_id"].item()
        orig_h, orig_w = targets[0]["orig_size"].tolist()
        images = images.to(device)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            outputs = lwdetr(images)
        pred_logits = outputs["pred_logits"][0]
        pred_boxes = outputs["pred_boxes"][0]

        scores, class_ids = pred_logits.sigmoid().max(-1)
        keep = scores > CONFIDENCE

        boxes_xyxy = (box_cxcywh_to_xyxy(pred_boxes[keep]).cpu().numpy() * np.array([orig_w, orig_h, orig_w, orig_h])).tolist()
        scores_list = scores[keep].cpu().tolist()
        class_ids_list = class_ids[keep].cpu().tolist()

        all_test_predictions[str(img_id)] = {
            "boxes_xyxy": boxes_xyxy,
            "confidence": scores_list,
            "class_id": class_ids_list,
            "orig_size": [orig_h, orig_w]
        }

with open(pred_out_file, "w", encoding="utf-8") as f:
    json.dump({
        "pipeline": PIPELINE_NAME,
        "mode": MODE_TAG,
        "num_classes": NUM_CLASSES,
        "class_names": class_names,
        "confidence_threshold": CONFIDENCE,
        "predictions": all_test_predictions
    }, f, indent=2)